# BFT65B - Part 5: Spark MLlib demo
Trains the baseline model of Part 5 (class-weighted logistic regression with median imputation and standardisation, fitted on the training
period only) with **Spark MLlib**, checks it against scikit-learn on the same data, and adds a random forest.

Data: `mllib_sample.parquet` (from `data_clean/`, made by `08_model_selection.py`): fold F3 of the validation scheme,
training rows 1985-2004 (all positives, 25% of negatives with weight 4) and validation rows 2006-2010.
**Runtime > Run all**; upload `mllib_sample.parquet` when asked.

In [1]:
%pip install -q pyspark

In [2]:
import re, subprocess
def java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        return int(re.search(r'version "(\d+)', out).group(1))
    except Exception:
        return 0
if java_major() < 17:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless", shell=True)
print("Java major version:", java_major())

Java major version: 21


In [3]:
import os, hashlib
def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()
if not os.path.exists("mllib_sample.parquet"):
    from google.colab import files
    files.upload()
print("mllib_sample.parquet", round(os.path.getsize("mllib_sample.parquet") / 1e6, 1), "MB   md5:", md5("mllib_sample.parquet"))
print("Compare size and md5 with what 08_model_selection.py printed on your computer.")

Saving mllib_sample.parquet to mllib_sample.parquet
mllib_sample.parquet 99.0 MB   md5: 82c5c697c56b0ab5a70f61a0f6e4f41e
Compare size and md5 with what 08_model_selection.py printed on your computer.


In [4]:
import time, statistics as st
import numpy as np, pandas as pd
import pyspark
from pyspark.sql import SparkSession, functions as F

RESULTS = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); RESULTS.append(s)

spark = (SparkSession.builder.master("local[*]").appName("bft65b-mllib")
         .config("spark.sql.shuffle.partitions", "8").config("spark.driver.memory", "6g")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
say(f"Spark {pyspark.__version__} local mode, {spark.sparkContext.defaultParallelism} cores")

raw = spark.read.parquet("mllib_sample.parquet")
FEATS = [c for c in raw.columns if c not in ("part", "y", "sample_weight")]
sdf = raw.select("part", F.col("y").cast("double").alias("y"), F.col("sample_weight").cast("double").alias("sample_weight"),
                 *[F.col(c).cast("double").alias(c) for c in FEATS])
n_tr, n_va = sdf.filter("part = 'train'").count(), sdf.filter("part = 'valid'").count()
n_pos_va = int(sdf.filter("part = 'valid'").agg(F.sum("y")).first()[0])
say(f"{len(FEATS)} features; train rows {n_tr:,}; validation rows {n_va:,}; positive validation rows {n_pos_va:,}")

Spark 4.0.5 local mode, 2 cores
47 features; train rows 268,874; validation rows 168,359; positive validation rows 1,580


## 1. Class weights (same definition for both libraries)

In [5]:
# 'balanced' class weights computed on the TRAINING rows only, multiplied by the sampling weight
tr_stats = sdf.filter("part = 'train'").groupBy("y").agg(F.sum("sample_weight").alias("W")).collect()
W = {r["y"]: r["W"] for r in tr_stats}
W_tot = W[0.0] + W[1.0]
factor = {0.0: W_tot / (2 * W[0.0]), 1.0: W_tot / (2 * W[1.0])}
sdf = sdf.withColumn("w", F.col("sample_weight") * F.when(F.col("y") == 1, factor[1.0]).otherwise(factor[0.0]))
train, valid = sdf.filter("part = 'train'"), sdf.filter("part = 'valid'")
W_sum = train.agg(F.sum("w")).first()[0]
say(f"Total training weight {W_sum:,.1f}; class factors: negative {factor[0.0]:.3f}, positive {factor[1.0]:.3f}")

Total training weight 1,046,981.0; class factors: negative 0.505, positive 55.075


## 2. MLlib pipeline: median imputation, standardisation, logistic regression (fitted on the training rows only)

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

C = 1.0
imp = Imputer(inputCols=FEATS, outputCols=[c + "_i" for c in FEATS], strategy="median")
asm = VectorAssembler(inputCols=[c + "_i" for c in FEATS], outputCol="raw")
scl = StandardScaler(inputCol="raw", outputCol="x", withMean=True, withStd=True)
lr = LogisticRegression(featuresCol="x", labelCol="y", weightCol="w", regParam=1.0 / (C * W_sum),
                        elasticNetParam=0.0, standardization=False, maxIter=300, tol=1e-9)
ev_roc = BinaryClassificationEvaluator(labelCol="y", rawPredictionCol="rawPrediction", metricName="areaUnderROC", numBins=0)
ev_pr = BinaryClassificationEvaluator(labelCol="y", rawPredictionCol="rawPrediction", metricName="areaUnderPR", numBins=0)

t0 = time.time()
model = Pipeline(stages=[imp, asm, scl, lr]).fit(train)          # every stage learns from `train` only
t_fit_spark = time.time() - t0
pred = model.transform(valid)
roc_spark, pr_spark = ev_roc.evaluate(pred), ev_pr.evaluate(pred)
say(f"MLlib logistic regression: fit {t_fit_spark:.1f} s; validation ROC-AUC {roc_spark:.4f}, PR-AUC (trapezoid) {pr_spark:.4f}")
coef_spark = np.array(model.stages[-1].coefficients)

MLlib logistic regression: fit 101.1 s; validation ROC-AUC 0.9641, PR-AUC (trapezoid) 0.4679


## 3. The same model in scikit-learn (same weights, same regularisation)

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, auc

pdf = pd.read_parquet("mllib_sample.parquet")
pdf["w"] = pdf["sample_weight"] * np.where(pdf["y"] == 1, factor[1.0], factor[0.0])
tr, va = pdf[pdf.part == "train"], pdf[pdf.part == "valid"]
t0 = time.time()
imp_sk = SimpleImputer(strategy="median").fit(tr[FEATS])
scl_sk = SkScaler().fit(imp_sk.transform(tr[FEATS]))
Xtr = scl_sk.transform(imp_sk.transform(tr[FEATS]))
clf = SkLR(C=C, max_iter=1000, tol=1e-9).fit(Xtr, tr["y"], sample_weight=tr["w"])
t_fit_sk = time.time() - t0
p = clf.predict_proba(scl_sk.transform(imp_sk.transform(va[FEATS])))[:, 1]
prec, rec, _ = precision_recall_curve(va["y"], p)
roc_sk, pr_trap_sk, ap_sk = roc_auc_score(va["y"], p), auc(rec, prec), average_precision_score(va["y"], p)
say(f"scikit-learn logistic regression: fit {t_fit_sk:.1f} s; validation ROC-AUC {roc_sk:.4f}, PR-AUC (trapezoid) {pr_trap_sk:.4f}, average precision {ap_sk:.4f}")
corr = np.corrcoef(coef_spark, clf.coef_[0])[0, 1]
say(f"Coefficients (standardised features): correlation {corr:.5f}, largest absolute difference {np.abs(coef_spark - clf.coef_[0]).max():.4f}")
say(f"Difference MLlib - scikit-learn: ROC-AUC {roc_spark - roc_sk:+.5f}, PR-AUC (trapezoid) {pr_spark - pr_trap_sk:+.5f}")
say("(Small differences are expected: Spark's median is approximate, its standard deviation uses n-1, and the optimisers stop differently.)")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


scikit-learn logistic regression: fit 91.0 s; validation ROC-AUC 0.9641, PR-AUC (trapezoid) 0.4680, average precision 0.4686
Coefficients (standardised features): correlation 0.99967, largest absolute difference 0.3358
Difference MLlib - scikit-learn: ROC-AUC -0.00000, PR-AUC (trapezoid) -0.00015
(Small differences are expected: Spark's median is approximate, its standard deviation uses n-1, and the optimisers stop differently.)


## 4. A distributed tree model: random forest in MLlib

In [8]:
rf = RandomForestClassifier(featuresCol="raw", labelCol="y", weightCol="w", numTrees=100, maxDepth=8, seed=42)
t0 = time.time()
rf_model = Pipeline(stages=[imp, asm, rf]).fit(train)
t_rf = time.time() - t0
pred_rf = rf_model.transform(valid)
say(f"MLlib random forest (100 trees, depth 8): fit {t_rf:.1f} s; validation ROC-AUC {ev_roc.evaluate(pred_rf):.4f}, PR-AUC (trapezoid) {ev_pr.evaluate(pred_rf):.4f}")
say(f"Baseline for comparison: the positive rate in the validation rows is {valid.agg(F.avg('y')).first()[0] * 100:.2f}% (PR-AUC of a random score).")

MLlib random forest (100 trees, depth 8): fit 127.6 s; validation ROC-AUC 0.9753, PR-AUC (trapezoid) 0.6137
Baseline for comparison: the positive rate in the validation rows is 0.94% (PR-AUC of a random score).


## 5. Copy the block below into the chat

In [9]:
print("\n".join(RESULTS))

Spark 4.0.5 local mode, 2 cores
47 features; train rows 268,874; validation rows 168,359; positive validation rows 1,580
Total training weight 1,046,981.0; class factors: negative 0.505, positive 55.075
MLlib logistic regression: fit 101.1 s; validation ROC-AUC 0.9641, PR-AUC (trapezoid) 0.4679
scikit-learn logistic regression: fit 91.0 s; validation ROC-AUC 0.9641, PR-AUC (trapezoid) 0.4680, average precision 0.4686
Coefficients (standardised features): correlation 0.99967, largest absolute difference 0.3358
Difference MLlib - scikit-learn: ROC-AUC -0.00000, PR-AUC (trapezoid) -0.00015
(Small differences are expected: Spark's median is approximate, its standard deviation uses n-1, and the optimisers stop differently.)
MLlib random forest (100 trees, depth 8): fit 127.6 s; validation ROC-AUC 0.9753, PR-AUC (trapezoid) 0.6137
Baseline for comparison: the positive rate in the validation rows is 0.94% (PR-AUC of a random score).
